# AMPS II · Week 07 — HW7
## Self-stress in a truss, section orientation and locking, convergence of the geometrically exact rod, and an LLM-generated rod script

**Due before Week 8 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q4 runs an LLM-generated script that is **meant** to be wrong.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Truss singularity, the locked Timoshenko element, rotations about a fixed axis (theory) | 20 |
| 2 | Linear: an indeterminate truss with lack of fit, a rectangular L-frame, locking vs slenderness | 30 |
| 3 | Finite rotations: element convergence on the helix, stiff rods and Newton, the source's load case | 30 |
| 4 | Debugging an LLM-generated rod script | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model modules from the lab (unchanged). Run the two cells once.

In [ ]:
%%writefile /tmp/fnx/frame.py
"""Linear trusses and 3D Timoshenko frames on 1D meshes embedded in 2D/3D, DOLFINx 0.11.
Members are straight lines between joints; each member is cut into n cells; the mesh is built with mesh.create_mesh (no Gmsh).
SERIAL ONLY when 3 or more members meet at a joint: in parallel (0.11) such a vertex is duplicated when its cells lie on
different ranks, and the structure falls apart. A chain (every vertex in at most 2 cells) is fine in parallel."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def lu(ksp):
    ksp.setType("preonly"); ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    return ksp

def line_mesh(X, members, n=1):
    """joints X (k x gdim), members = pairs of joint indices, each member cut into n cells.
    Returns the mesh and the member of every (local) cell."""
    X = np.asarray(X, dtype=float); gdim = X.shape[1]
    pts, cells, owner = [p for p in X], [], []
    for m, (i, j) in enumerate(members):
        prev = i
        for k in range(1, n + 1):
            if k < n:
                pts.append(X[i] + (X[j] - X[i]) * k / n); cur = len(pts) - 1
            else:
                cur = j
            cells.append((prev, cur)); owner.append(m); prev = cur
    if comm.rank == 0:
        x, c = np.array(pts), np.array(cells, dtype=np.int64)
    else:
        x, c = np.zeros((0, gdim)), np.zeros((0, 2), dtype=np.int64)
    msh = mesh.create_mesh(comm, c, ufl.Mesh(basix.ufl.element("Lagrange", "interval", 1, shape=(gdim,))), x)
    msh.topology.create_connectivity(0, 1)                     # needed by locate_dofs_topological on vertices
    return msh, np.array(owner, dtype=int)[msh.topology.original_cell_index]

def tangent(msh):
    """unit tangent of the cells, from the Jacobian of the 1D cell (its sign depends on the cell orientation: use it squared)"""
    J = ufl.Jacobian(msh)
    t = ufl.as_vector([J[i, 0] for i in range(msh.geometry.dim)])
    return t / ufl.sqrt(ufl.dot(t, t))

def vertex_dofs(V, x0, tol=1e-9):
    """DOFs of the (sub)space V at the joint x0. V may be a pair (V.sub(i), collapsed) as for dirichletbc with a Function."""
    msh = (V[0] if isinstance(V, tuple) else V).mesh
    x0 = np.asarray(x0, dtype=float)
    vs = mesh.locate_entities(msh, 0, lambda p: np.all(np.abs(p[:len(x0)].T - x0) < tol, axis=1))
    return fem.locate_dofs_topological(V, 0, vs)

def _add_point_values(b, V, x0, vals, nown):
    for c, f in enumerate(vals):
        if f != 0.0:
            for d in vertex_dofs(V.sub(c), x0):
                if d < nown:
                    b.array[d] += f


class Truss:
    """pin-jointed truss, u in P1. One cell per member: a member cut into more cells leaves the transverse motion of its inner
    nodes without stiffness (singular matrix). ES: one value or one per member; eps0: initial (lack-of-fit / thermal) strain
    per member, N = ES (eps - eps0)."""
    def __init__(self, X, members, ES, n=1, eps0=0.0):
        self.msh, self.member = line_mesh(X, members, n)
        msh = self.msh; gdim = self.gdim = msh.geometry.dim; nm = len(members)
        V = self.V = fem.functionspace(msh, ("Lagrange", 1, (gdim,)))
        t = tangent(msh)
        self.eps = lambda w: ufl.dot(ufl.dot(ufl.grad(w), t), t)
        Q = fem.functionspace(msh, ("DG", 0))
        self.ES, self.eps0 = fem.Function(Q), fem.Function(Q)
        self.ES.x.array[:] = np.broadcast_to(np.asarray(ES, dtype=float), (nm,))[self.member]
        self.eps0.x.array[:] = np.broadcast_to(np.asarray(eps0, dtype=float), (nm,))[self.member]
        u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        dx = ufl.Measure("dx", domain=msh)
        self.a = fem.form(self.ES * self.eps(u) * self.eps(v) * dx)
        self.L = fem.form(self.ES * self.eps0 * self.eps(v) * dx)
        self.bcs, self.loads = [], []
        self.u = fem.Function(V)

    def fix(self, x0, comps=None):
        """fix the components comps (None = all) of the joint at x0"""
        if comps is None:
            self.bcs.append(fem.dirichletbc(np.zeros(self.gdim), vertex_dofs(self.V, x0), self.V))
        else:
            for c in comps:
                self.bcs.append(fem.dirichletbc(0.0, vertex_dofs(self.V.sub(c), x0), self.V.sub(c)))

    def load(self, x0, F):
        self.loads.append((np.asarray(x0, dtype=float), np.asarray(F, dtype=float)))

    def solve(self):
        A = assemble_matrix(self.a, bcs=self.bcs); A.assemble()
        b = assemble_vector(self.L)
        nown = self.V.dofmap.index_map.size_local * self.gdim
        for x0, F in self.loads:
            _add_point_values(b, self.V, x0, F, nown)
        apply_lifting(b, [self.a], [self.bcs]); b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(b, self.bcs)
        k = lu(PETSc.KSP().create(comm)); k.setOperators(A); k.solve(b, self.u.x.petsc_vec); self.u.x.scatter_forward()
        return k.getConvergedReason()

    def forces(self):
        """normal force N = ES (eps - eps0) of every member, in member order (serial)"""
        Q = self.ES.function_space; N = fem.Function(Q)
        N.interpolate(fem.Expression(self.ES * (self.eps(self.u) - self.eps0), Q.element.interpolation_points))
        out = np.zeros(self.member.max() + 1)
        out[self.member] = N.x.array[:len(self.member)]
        return out


class Frame:
    """3D Timoshenko frame, unknowns (u, theta). Local frame per member: a1 = up x t / |up x t|, a2 = t x a1, with a reference
    vector 'up' (one vector, or one per member) that must NOT be parallel to the member. Section dict: ES, kGS1, kGS2, EI1, EI2, GJ,
    with I1 = int x2^2 dA (bending about a1 = deflection along a2) and I2 = int x1^2 dA.
    elem = "P1" (P1/P1, full integration: shear locking), "P1-RI" (one-point integration of the shear terms), "P2P1" (u in P2)."""
    def __init__(self, X, members, sec, n=10, elem="P1-RI", up=(0.0, 0.0, 1.0)):
        self.msh, self.member = line_mesh(X, members, n)
        msh = self.msh; cell = msh.basix_cell()
        V = self.V = fem.functionspace(msh, basix.ufl.mixed_element([
            basix.ufl.element("Lagrange", cell, 2 if elem == "P2P1" else 1, shape=(3,)), basix.ufl.element("Lagrange", cell, 1, shape=(3,))]))
        t = self.t = tangent(msh)
        up = np.asarray(up, dtype=float)
        if up.ndim == 2:                                       # one reference vector per member: a DG0 vector field
            U = fem.Function(fem.functionspace(msh, ("DG", 0, (3,))))
            U.x.array[:] = up[self.member].ravel(); upv = U
        else:
            upv = ufl.as_vector(up)
        a1 = ufl.cross(upv, t); a1 = a1 / ufl.sqrt(ufl.dot(a1, a1)); a2 = ufl.cross(t, a1)
        self.a1, self.a2 = a1, a2
        s = self.sec = {k: fem.Constant(msh, float(v)) for k, v in sec.items()}
        tg = lambda w: ufl.dot(ufl.grad(w), t)                  # d/ds along the member
        def strains(u, th):
            return dict(N=ufl.dot(tg(u), t), T=ufl.dot(tg(th), t), M1=ufl.dot(tg(th), a1), M2=ufl.dot(tg(th), a2),
                        Q1=ufl.dot(tg(u), a1) - ufl.dot(th, a2), Q2=ufl.dot(tg(u), a2) + ufl.dot(th, a1))
        self.strains = strains
        self.stiff = dict(N="ES", T="GJ", M1="EI1", M2="EI2", Q1="kGS1", Q2="kGS2")
        dx = ufl.Measure("dx", domain=msh)
        dxs = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 0}) if elem == "P1-RI" else dx
        (u, th), (v, ph) = ufl.TrialFunctions(V), ufl.TestFunctions(V)
        e, ev = strains(u, th), strains(v, ph)
        self.a = fem.form(sum(s[self.stiff[k]] * e[k] * ev[k] for k in ("N", "T", "M1", "M2")) * dx
                          + sum(s[self.stiff[k]] * e[k] * ev[k] for k in ("Q1", "Q2")) * dxs)
        self.bcs, self.loads = [], []
        self.w = fem.Function(V)

    def clamp(self, x0):
        for i in (0, 1):
            Vc, _ = self.V.sub(i).collapse()
            self.bcs.append(fem.dirichletbc(fem.Function(Vc), vertex_dofs((self.V.sub(i), Vc), x0), self.V.sub(i)))

    def load(self, x0, F=(0, 0, 0), M=(0, 0, 0)):
        self.loads.append((np.asarray(x0, dtype=float), np.asarray(F, dtype=float), np.asarray(M, dtype=float)))

    def solve(self):
        A = assemble_matrix(self.a, bcs=self.bcs); A.assemble()
        b = self.w.x.petsc_vec.duplicate(); b.set(0.0)
        nown = self.V.dofmap.index_map.size_local * self.V.dofmap.index_map_bs
        for x0, F, M in self.loads:
            _add_point_values(b, self.V.sub(0), x0, F, nown); _add_point_values(b, self.V.sub(1), x0, M, nown)
        apply_lifting(b, [self.a], [self.bcs]); b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(b, self.bcs)
        k = lu(PETSc.KSP().create(comm)); k.setOperators(A); k.solve(b, self.w.x.petsc_vec); self.w.x.scatter_forward()
        return k.getConvergedReason()

    def at(self, x0):
        """(u, theta) at the joint x0 (serial)"""
        return tuple(np.array([self.w.x.array[vertex_dofs(self.V.sub(i).sub(c), x0)[0]] for c in range(3)]) for i in (0, 1))

    def resultants(self):
        """section forces per cell (DG0 at the cell midpoint): dict N, Q1, Q2, T, M1, M2 and the midpoints 'x' (serial)"""
        Q = fem.functionspace(self.msh, ("DG", 0)); u, th = self.w.split(); e = self.strains(u, th); out = {}
        for k in e:
            f = fem.Function(Q); f.interpolate(fem.Expression(self.sec[self.stiff[k]] * e[k], Q.element.interpolation_points))
            out[k] = f.x.array.copy()
        X = fem.Function(fem.functionspace(self.msh, ("DG", 0, (3,))))
        X.interpolate(fem.Expression(ufl.SpatialCoordinate(self.msh), X.function_space.element.interpolation_points))
        out["x"] = X.x.array.reshape(-1, 3).copy()
        return out

def rect(b, h, E, nu, kappa=5 / 6):
    """rectangle, b along a1 and h along a2: I1 = b h^3/12 (deflection along a2), I2 = h b^3/12; St-Venant J from the series"""
    G = E / (2 * (1 + nu)); A = b * h; p, q = max(b, h), min(b, h)
    J = p * q**3 * (1 / 3 - 64 / np.pi**5 * q / p * sum(np.tanh((2 * k + 1) * np.pi * p / (2 * q)) / (2 * k + 1)**5 for k in range(50)))
    return dict(ES=E * A, kGS1=kappa * G * A, kGS2=kappa * G * A, EI1=E * b * h**3 / 12, EI2=E * h * b**3 / 12, GJ=G * J)

def circ(r, E, nu):
    """solid circle: I = pi r^4/4, J = pi r^4/2, kappa = 6 (1 + nu)/(7 + 6 nu) (Cowper)"""
    G = E / (2 * (1 + nu)); A = np.pi * r**2; I = np.pi * r**4 / 4; k = 6 * (1 + nu) / (7 + 6 * nu)
    return dict(ES=E * A, kGS1=k * G * A, kGS2=k * G * A, EI1=E * I, EI2=E * I, GJ=G * 2 * I)

In [ ]:
%%writefile /tmp/fnx/rod.py
"""Geometrically exact (Simo-Reissner) rod in 3D, DOLFINx 0.11. Total formulation with the rotation vector theta:
Lambda = exp(Skew theta) (Rodrigues),  Gamma = Lambda^T (t0 + u') - t0  (axial and shear strain),  K = T(theta)^T theta'
(curvature and twist; T = the tangent operator of the exponential map),  energy 1/2 Gamma.C_N Gamma + 1/2 K.C_M K,
residual = ufl.derivative of the potential, Newton = SNES. Straight rod on the x axis, clamped at x = 0, dead tip force F and
a tip moment M with the work M . theta(L) (exact for a rotation about a fixed axis, as in all examples here).
elem: "P1" (u, theta in P1, full integration: locks), "P1-RI" (one-point rule for the force term), "P2P1" (u in P2, theta in P1,
full integration), "P2P1-RI" (two-point rule for the force term)."""
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def skew(n):
    return ufl.as_matrix([[0, -n[2], n[1]], [n[2], 0, -n[0]], [-n[1], n[0], 0]])

def expmap(th, eps=1e-12):
    """Lambda and T of the rotation vector th (|th| -> 0 regularised by eps)"""
    x = ufl.sqrt(ufl.dot(th, th) + eps); S = skew(th); I = ufl.Identity(3)
    Lam = I + ufl.sin(x) / x * S + (1 - ufl.cos(x)) / x**2 * S * S
    T = I + (1 - ufl.cos(x)) / x**2 * S + (x - ufl.sin(x)) / x**3 * S * S
    return Lam, T

def rodrigues(th):
    """NumPy: the rotation matrix of a rotation vector"""
    th = np.asarray(th, dtype=float); x = np.linalg.norm(th)
    if x < 1e-14:
        return np.eye(3)
    S = np.array([[0, -th[2], th[1]], [th[2], 0, -th[0]], [-th[1], th[0], 0]])
    return np.eye(3) + np.sin(x) / x * S + (1 - np.cos(x)) / x**2 * S @ S

def rod_mesh(L, n):
    """n cells on [0, L] x {0} x {0}: a 1D mesh in 3D (u and theta get 3 components). A chain: parallel is fine."""
    if comm.rank == 0:
        x = np.c_[np.linspace(0, L, n + 1), np.zeros(n + 1), np.zeros(n + 1)]
        c = np.c_[np.arange(n), np.arange(1, n + 1)].astype(np.int64)
    else:
        x, c = np.zeros((0, 3)), np.zeros((0, 2), dtype=np.int64)
    return mesh.create_mesh(comm, c, ufl.Mesh(basix.ufl.element("Lagrange", "interval", 1, shape=(3,))), x)

class Rod:
    """C_N = diag(ES, GS, GS), C_M = diag(GJ, EI, EI) in the material frame (e1 = the rod axis)."""
    count = 0                       # a new options prefix per rod: PETSc keeps options of a prefix for every later solver using it
    def __init__(self, L, n, ES, GS, GJ, EI, elem="P2P1-RI", linesearch="basic", opts=None):
        self.L = L
        msh = self.msh = rod_mesh(L, n); cell = msh.basix_cell()
        V = self.V = fem.functionspace(msh, basix.ufl.mixed_element([
            basix.ufl.element("Lagrange", cell, 2 if elem.startswith("P2") else 1, shape=(3,)), basix.ufl.element("Lagrange", cell, 1, shape=(3,))]))
        self.w = fem.Function(V); u, th = ufl.split(self.w)
        t0 = ufl.as_vector([1.0, 0.0, 0.0])
        Lam, T = expmap(th)
        self.Gam = Lam.T * (t0 + u.dx(0)) - t0                  # the mesh is the x axis: d/ds = d/dx
        self.Kap = T.T * th.dx(0)
        CN, CM = ufl.diag(ufl.as_vector([ES, GS, GS])), ufl.diag(ufl.as_vector([GJ, EI, EI]))
        qN = {"P1": 4, "P1-RI": 1, "P2P1": 4, "P2P1-RI": 2}[elem]        # points of the rule for the force term: 1 (deg 1), 2 (deg 2)
        dxN = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": qN})
        dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 4})
        tip = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], L))
        ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 0, tip, np.ones(len(tip), dtype=np.int32)))
        self.F, self.M = fem.Constant(msh, np.zeros(3)), fem.Constant(msh, np.zeros(3))
        self.Pi_int = 0.5 * ufl.dot(self.Gam, CN * self.Gam) * dxN + 0.5 * ufl.dot(self.Kap, CM * self.Kap) * dx
        Pi = self.Pi_int - ufl.dot(self.F, u) * ds(1) - ufl.dot(self.M, th) * ds(1)
        root = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], 0.0))
        self.bcs = []
        for i in (0, 1):
            Vc, _ = V.sub(i).collapse()
            self.bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_topological((V.sub(i), Vc), 0, root), V.sub(i)))
        o = {"snes_type": "newtonls", "snes_linesearch_type": linesearch, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50,
             "snes_divergence_tolerance": 1e12,      # the first Newton update of a stiff rod is the LINEAR solution: the residual grows
             "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}     # x 1e4 before Newton converges
        o.update(opts or {})
        Rod.count += 1
        self.problem = NonlinearProblem(ufl.derivative(Pi, self.w, ufl.TestFunction(V)), self.w, bcs=self.bcs,
                                        petsc_options_prefix=f"rod{Rod.count}_", petsc_options=o)
        self.tip_forms = [fem.form(self.w.sub(i)[c] * ds(1)) for i in (0, 1) for c in range(3)]
        self.energy_form = fem.form(self.Pi_int)

    def solve(self, F=None, M=None):
        """set the TOTAL tip loads and solve from the current state; returns (SNES reason, Newton iterations)"""
        if F is not None:
            self.F.value[:] = F
        if M is not None:
            self.M.value[:] = M
        self.problem.solve()
        s = self.problem.solver
        return s.getConvergedReason(), s.getIterationNumber()

    def tip(self):
        """(u, theta) at x = L"""
        v = np.array([comm.allreduce(fem.assemble_scalar(f), op=MPI.SUM) for f in self.tip_forms])
        return v[:3], v[3:]

    def energy(self):
        return comm.allreduce(fem.assemble_scalar(self.energy_form), op=MPI.SUM)

    def centerline(self):
        """deformed centre line x + u at the P1 vertices, sorted by x (serial)"""
        Vu, m = self.V.sub(0).collapse(); X = Vu.tabulate_dof_coordinates()
        u = self.w.x.array[m].reshape(-1, 3); i = np.argsort(X[:, 0])
        return X[i] + u[i]

def circle_tip(L, EI, M):
    """pure end moment M about -y: a circular arc of curvature M/EI (any ES, GS: no force, no strain): tip displacement"""
    p = M * L / EI
    return np.array([L * np.sin(p) / p - L, 0.0, L * (1 - np.cos(p)) / p])

def helix_tip(L, EI, m):
    """isotropic rod (EI = GJ) under a dead end moment m: k = m/EI is constant in space, theta(s) = s k, the centre line is a
    helix about m. Tip displacement int_0^L Lambda(s k) e1 ds - L e1 (closed form)."""
    k = np.asarray(m, dtype=float) / EI; a = np.linalg.norm(k); n = k / a; e1 = np.array([1.0, 0, 0])
    par = np.dot(n, e1) * n; perp = e1 - par
    r = par * L + perp * np.sin(a * L) / a + np.cross(n, perp) * (1 - np.cos(a * L)) / a
    return r - L * e1

def reissner(F, L, EI, ES=np.inf, GS=np.inf):
    """planar cantilever, dead tip load F downward, extensible and shear deformable (Reissner); ES = GS = inf: the elastica.
    Shooting on the tip abscissa X; returns the tip (x, downward w) and the tip angle. Needs scipy."""
    from scipy.integrate import solve_ivp
    from scipy.optimize import brentq
    def run(X):
        def f(s, y):
            x, z, p = y
            e, g = F * np.sin(p) / ES, -F * np.cos(p) / GS
            return [(1 + e) * np.cos(p) + g * np.sin(p), -(1 + e) * np.sin(p) + g * np.cos(p), F * (X - x) / EI]
        return solve_ivp(f, [0, L], [0, 0, 0], rtol=1e-12, atol=1e-13).y[:, -1]
    X = brentq(lambda X: run(X)[0] - X, 0, 1.01 * L)
    x, z, p = run(X)
    return x, -z, p

## Q1 — Theory (20 pts)

**(a) (6)** A straight truss member cut into two P1 cells, axial stiffness $ES$. Write the $6\times6$ (2D) stiffness of the two cells and show that the middle node has no stiffness
across the member. Why does the method of joints need no $ES$ for the Warren truss, and what is the condition $2j=m+r$?

**(b) (8)** A Timoshenko cantilever with P1 $w$ and P1 $\theta$ on cells of length $\ell$. Show that with exact (two-point) integration of $\kappa GA(w'-\theta)^2$ a cell carrying a linearly varying
$\theta$ stores the extra energy $\kappa GA\,\ell^3\theta'^2/24$, i.e. the bending stiffness becomes $EI_{\rm eff}=EI+\kappa GA\,\ell^2/12$. Check the prediction
$w/w_{\rm Tim}\approx EI/EI_{\rm eff}$ against the lab (§3, $n$ = 10 and 100). Why does one-point integration remove the extra term?

**(c) (6)** For $\boldsymbol\theta(s)=\varphi(s)\,\mathbf n$ with a fixed unit vector $\mathbf n$, show $T(\boldsymbol\theta)^\top\boldsymbol\theta'=\varphi'\mathbf n$ and $T(\boldsymbol\theta)^\top\mathbf n=\mathbf n$. Conclude that (i) a pure end moment gives a
circle, (ii) the helix of the lab has $\boldsymbol\theta(s)=s\,\mathbf m/EI$, and (iii) the load term $\mathbf M\cdot\boldsymbol\theta(L)$ has the same equilibrium as the true work of a dead moment,
$\int\mathbf M\cdot T\,\delta\boldsymbol\theta$, in these cases.

> *your answer*

## Q2 — Linear trusses and frames (30 pts)

**(a) (12)** Add a member from joint 2 $(2,0)$ to joint 10 $(3.5,1)$ to the Warren truss (it crosses the diagonal 3–9 without a joint): now $m+r=27>2j=26$. The new member is
$\delta=1$ mm **too short** and is forced into place (`eps0 = -delta / length` for it). Solve (i) the loads alone, (ii) the lack of fit alone, (iii) both. Compare (i) with the determinate truss, check
(iii) = (i) + (ii), and check (ii) with the force method in NumPy (one redundant). What happens to (ii) if every $ES$ is doubled? And to a lack of fit in the **determinate** truss?

In [ ]:
%%fenicsx -np 1
# TODO: Q2(a) — the indeterminate truss with lack of fit

> *your answer (Q2a)*

**(b) (10)** The L-frame of the lab ($a=2$, $b=1$, $F_z=-1$ kN) with a **rectangular** section $0.1\times0.2$ m, $E=210$ GPa, $\nu=0.3$, `up` $=\mathbf e_z$. With `rect(0.1, 0.2, ...)` the 0.2 m side lies
along $\mathbf a_2=\mathbf e_z$ (upright); with `rect(0.2, 0.1, ...)` the section is flat. Compute the tip deflection for both, $n=32$, P1-RI, and compare with the exact formula with the right $I$,
$J$ (Saint-Venant, from `rect`) and $\kappa GA$. Which term changes most between the two orientations, and why does torsion barely care?

In [ ]:
%%fenicsx -np 1
# TODO: Q2(b) — section orientation

> *your answer (Q2b)*

**(c) (8)** Locking against slenderness. The lab's cantilever with $h$ = 2, 0.5, 0.1 m ($L=10$, $b=0.3$), $n=10$: P1 (full) and P1-RI tip deflection over Timoshenko. Compare P1 with your
Q1(b) prediction $EI/EI_{\rm eff}$, and P1-RI with $1-\frac{1}{4n^2}\cdot\frac{w_{EB}}{w_{\rm Tim}}$. Then P1-RI for $n$ = 2, 4, 8, 16: the convergence rate.

In [ ]:
%%fenicsx -np 1
# TODO: Q2(c) — locking vs slenderness

> *your answer (Q2c)*

## Q3 — The geometrically exact rod (30 pts)

The lab's rod ($L=10$, $EI=GJ=100$, $ES=GS=10^4$).

**(a) (12)** Convergence on the helix ($|\mathbf m|L/EI=3\pi$, $\mathbf m\parallel(0.6,0,0.8)$, 10 load steps): `P1-RI`, `P2P1-RI` and `P2P1` (full integration, the source's element) for $n$ = 10, 20, 40, 80. Print the
SNES reason of the last step, $|\mathbf u-\mathbf u_{\rm ex}|$ at the tip, and the rates. Explain each element's behaviour.

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a) — element convergence on the helix

> *your answer (Q3a)*

**(b) (10)** A stiff rod: $ES=GS=10^7$ (very slender, $ES\,L^2/EI=10^7$), the tip load of the lab up to $\alpha=10$ in 10 steps, `P2P1-RI`, $n=40$. Run it once with PETSc's default divergence
tolerance (`opts={"snes_divergence_tolerance": 1e4}`) and once with the module's setting. Print, for the first load step, the residual norm after each Newton iteration
(`opts={"snes_monitor": None}` prints it), the reasons, and the final $w/L$ against `reissner`. Why does the default fail, and is the large first residual a sign of trouble?

In [ ]:
%%fenicsx -np 1
# TODO: Q3(b) — a stiff rod and SNES's divergence test

> *your answer (Q3b)*

**(c) (8)** The source's load case: tip force $F=50\,t$ along $+z$ **and** tip moment $M=100\cdot2\pi\,t$ about $+y$, $t\in(0,1]$ in 100 steps, applied as **total** loads (`P2P1-RI`, $n=40$). Print the tip
$(x, y, z)$ at $t$ = 0.01, 0.25, 0.5, 1 and the largest $|u_y|$. Compare $t=0.01$ with the source ($u_z=-1.4997$ m, called "10 % error from shear locking" against $FL^3/3EI=1.6667$ m) and with
the linear response to **both** loads. Is the tip path a helix?

In [ ]:
%%fenicsx -np 2
# TODO: Q3(c) — the source's load case, done right

> *your answer (Q3c)*

## Q4 — Debugging an LLM-generated rod script (20 pts)

A student asked an LLM for "a self-contained DOLFINx 0.11 script for a geometrically exact 3D beam (rotation vector, exponential map), cantilever $L=10$, $EI=GJ=100$,
$ES=GS=10^4$; roll it up with a tip moment and bend it with a tip force; print the tip". The script below runs, converges and prints plausible numbers. It has **four** defects.

**(a) (4)** Run it and record the output.

**(b) (8)** List the four defects and the test that exposes each (the exact circle, the orthogonality of $\Lambda$, the elastica, refinement, …).

**(c) (8)** Correct the script (second cell) and verify it against `circle_tip` and `reissner` (copy the formulas, or import `rod.py`).

In [ ]:
%%fenicsx -np 1
# LLM-generated (as received)
import numpy as np, ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

comm = MPI.COMM_WORLD
L, EI, GJ, ES, GS, n = 10.0, 100.0, 100.0, 1e4, 1e4, 40
x = np.c_[np.linspace(0, L, n + 1), np.zeros(n + 1), np.zeros(n + 1)]
cells = np.c_[np.arange(n), np.arange(1, n + 1)].astype(np.int64)
msh = mesh.create_mesh(comm, cells, ufl.Mesh(basix.ufl.element("Lagrange", "interval", 1, shape=(3,))), x)
P1 = basix.ufl.element("Lagrange", "interval", 1, shape=(3,))
V = fem.functionspace(msh, basix.ufl.mixed_element([P1, P1]))
w = fem.Function(V); u, th = ufl.split(w)

def skew(a):
    return ufl.as_matrix([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
phi = ufl.sqrt(ufl.dot(th, th) + 1e-12); S = skew(th); I3 = ufl.Identity(3)
Lam = I3 + ufl.sin(phi) / phi * S + (1 - ufl.cos(phi)) / phi * S * S              # Rodrigues
T = I3 + (1 - ufl.cos(phi)) / phi**2 * S + (phi - ufl.sin(phi)) / phi**3 * S * S
t0 = ufl.as_vector([1.0, 0.0, 0.0])
Gam = Lam * (t0 + u.dx(0)) - t0                                                 # axial + shear strain
Kap = T.T * th.dx(0)                                                            # curvature
CN, CM = ufl.diag(ufl.as_vector([ES, GS, GS])), ufl.diag(ufl.as_vector([GJ, EI, EI]))
tip = mesh.locate_entities_boundary(msh, 0, lambda X: np.isclose(X[0], L))
ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 0, tip, np.ones(len(tip), dtype=np.int32)))
Fc, Mc = fem.Constant(msh, np.zeros(3)), fem.Constant(msh, np.zeros(3))
Pi = (0.5 * ufl.dot(Gam, CN * Gam) + 0.5 * ufl.dot(Kap, CM * Kap)) * ufl.dx - ufl.dot(Fc, u) * ds(1) - ufl.dot(Mc, th) * ds(1)
root = mesh.locate_entities_boundary(msh, 0, lambda X: np.isclose(X[0], 0.0))
bcs = []
for i in (0, 1):
    Vc, _ = V.sub(i).collapse()
    bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_topological((V.sub(i), Vc), 0, root), V.sub(i)))
problem = NonlinearProblem(ufl.derivative(Pi, w, ufl.TestFunction(V)), w, bcs=bcs, petsc_options_prefix="llm_",
                           petsc_options={"snes_type": "newtonls", "snes_linesearch_type": "basic", "snes_rtol": 1e-10, "ksp_type": "preonly", "pc_type": "lu",
                                          "pc_factor_mat_solver_type": "mumps"})
tipf = [fem.form(w.sub(0)[c] * ds(1)) for c in range(3)]
def tip_u():
    return np.array([fem.assemble_scalar(f) for f in tipf])

nsteps = 10
for k in range(1, nsteps + 1):                                                 # roll up: half a circle, M = pi EI / L
    Mc.value[:] = (0.0, -np.pi * EI / L / nsteps, 0.0)
    problem.solve()
print(f"moment pi EI/L: tip = {np.round(tip_u(), 4)}  (SNES {problem.solver.getConvergedReason()})")
w.x.array[:] = 0.0; Mc.value[:] = 0.0
for k in range(1, nsteps + 1):                                                 # bend: F L^2 / EI = 5
    Fc.value[:] = (0.0, 0.0, -5 * EI / L**2 / nsteps)
    problem.solve()
print(f"tip force 5 EI/L^2: tip = {np.round(tip_u(), 4)}  (SNES {problem.solver.getConvergedReason()})")

In [ ]:
%%fenicsx -np 1
# TODO: Q4(c) — the corrected script and its verification

> *your list of defects and tests (Q4b), and your verification (Q4c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …